In [1]:
# =============================================================================
# STEP 0: Force reload modules (run this first after code changes!)
# =============================================================================
import importlib
import src.data_loader
import src.features
import src.models.base
import src.models.goals
import src.models.assists
import src.models.minutes
import src.models.defcon
import src.models.clean_sheet
import src.models.bonus
import src.models.cards
import src.models.saves
import src.pipeline
import src.viz

importlib.reload(src.data_loader)
importlib.reload(src.features)
importlib.reload(src.models.base)
importlib.reload(src.models.goals)
importlib.reload(src.models.assists)
importlib.reload(src.models.minutes)
importlib.reload(src.models.defcon)
importlib.reload(src.models.clean_sheet)
importlib.reload(src.models.bonus)
importlib.reload(src.models.cards)
importlib.reload(src.models.saves)
importlib.reload(src.pipeline)
importlib.reload(src.viz)

print("All modules reloaded.")

All modules reloaded.


In [2]:
# =============================================================================
# STEP 1: Update Data (optional - only if you need fresh gameweek data)
# =============================================================================
#!python scrape_update_data.py --gameweek 29
# !python scrape_update_data.py --auto

In [3]:
# =============================================================================
# STEP 2: Run the Pipeline
# =============================================================================
from src.pipeline import FPLPipeline
from src.experiment_log import clear_experiments

pipeline = FPLPipeline('data', n_sims=5000)
pipeline.load_data()
pipeline.compute_features()

# Clear old experiments (old MAE was not using 25/26 actuals with bonus)
clear_experiments('data')
print("Cleared old experiment history (incompatible MAE calculation)")

# Full hyperparameter tuning (all models, 2025/26 holdout)
pipeline.tune(
    n_iter=100,
    use_subprocess=True,
    test_season='2025/2026',
    description='baseline: full tune, 5k sims, 25/26 holdout inc-bonus MAE',
    test_start_gw=20,
)

# Train final models on all data
pipeline.train()

# Predict
predictions = pipeline.predict(gameweek=31, season='2025/2026')

LOADING DATA
Loading player stats from: player_stats.csv
  Loaded 83,766 player-match records
  Seasons: ['2018/2019', '2019/2020', '2020/2021', '2021/2022', '2022/2023', '2023/2024', '2024/2025', '2025/2026']
Loaded 2,962 fixtures
Filtered to seasons: ['2020/2021', '2021/2022', '2022/2023', '2023/2024', '2024/2025', '2025/2026']
Current season (2025/2026): 521 active players
Final dataset: 63,493 records
Fetching actual FPL points for GW 1-30 (30 GWs)...
  Cache has 30 GWs already
  Cached to data\fpl_actual_points.csv
  Total: 23,086 player-gameweek records
  FPL card merge: 5,456 rows matched, 5,456 with card data, 395 total yellows, 42,753 DGW rows excluded

COMPUTING FEATURES
Computing rolling features...


c:\Users\dpfin\OneDrive\Desktop\projecting_fpl_v2\src\features.py:232: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'defcon_roll{window}'] = df.groupby('player_id')['defcon'].transform(
c:\Users\dpfin\OneDrive\Desktop\projecting_fpl_v2\src\features.py:232: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'defcon_roll{window}'] = df.groupby('player_id')['defcon'].transform(
c:\Users\dpfin\OneDrive\Desktop\projecting_fpl_v2\src\features.py:235: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result o

  Computed 136 rolling/lifetime features
Cleared old experiment history (incompatible MAE calculation)

TUNING HYPERPARAMETERS WITH HOLDOUT TEST SET

Data split (season=2025/2026):
  Train: 60,160 samples (['2020/2021', '2021/2022', '2022/2023', '2023/2024', '2024/2025', '2025/2026'])
  Test:  3,320 samples (['2025/2026'])
    2025/2026: GW20.0-GW31.0

------------------------------------------------------------
PHASE 1: Hyperparameter + Feature Selection Tuning (5-fold CV)
------------------------------------------------------------

Tuning MINUTES (two-stage, 100 trials each)...
  Total played: 60,160

  [1/3] StarterClassifier (32 features, 71.5% starters)
    Pre-computing feature rankings...


C:\Users\dpfin\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Best trial: 91. Best value: 0.442467: 100%|██████████| 100/100 [02:40<00:00,  1.60s/it]


    Best CV LogLoss: 0.4425
    Feature method: xgb_gain, selected 19/32 features

  [2/3] StarterMinutesModel (20 features, 43,043 samples)
    Pre-computing feature rankings...


...


Model           Metric          Test Value   MAE         
------------------------------------------------------
MINUTES         Huber Loss      143.0047     17.7220     
  Classifier    AUC             0.8026       LogLoss 0.7688
  Starter Reg   MAE             12.0361     
  Sub Reg       MAE             30.5434     
GOALS           Poisson Dev     0.3731       0.1531      
ASSISTS         Poisson Dev     0.3304       0.1195      
DEFCON          Poisson Dev     2.3040       2.7577      
SAVES           MAE             1.4358       1.4358      
CLEAN_SHEET     Poisson Dev     1.0921       0.8775      
------------------------------------------------------
FPL POINTS      ex-bonus MAE    1.7236      
                inc-bonus MAE   1.7794      
BONUS           MAE             0.2205      

(Lower is better for all metrics)
ex-bonus: FPL points excluding bonus on both sides
inc-bonus: FPL points with bonus on both sides (pred bonus vs actual bonus)

Experiment logged: 20260319_095340_

C:\Users\dpfin\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\xgboost\core.py:158: UserWarning: [09:53:40] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


  StarterMinutesModel: 45,343 samples, mean=85.9
  SubMinutesModel: 18,137 samples, mean=22.0
  Combined MAE: 15.3
Training CleanSheetModel (Goals Against) on 4,006 team-matches (15 features, tuned selection)...
  Avg goals conceded: 1.375
  Actual CS rate: 27.5%
  Prior lambda range: 0.30 - 4.00
  MAE (goals against): 0.941
  Predicted CS prob (mean): 30.5%
  Predicted 2+ conceded prob (mean): 37.5%
  Predicted CS prob range: 2.3% - 67.6%

Generating leak-free predicted team goals (TimeSeriesSplit OOF)...


c:\Users\dpfin\OneDrive\Desktop\projecting_fpl_v2\src\pipeline.py:279: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  self.df['opponent_norm'] = self.df['opponent'].apply(normalize_name)


  Mapped pred_team_goals to 63,493/63,493 player rows
  Mean pred_team_goals: 1.394
Training GoalsModel on 63,480 samples (37 features, tuned selection)...
  Target mean: 0.0951
  MAE: 0.1552
Training AssistsModel on 63,480 samples (27 features, tuned selection)...
  Target mean: 0.0678
  MAE: 0.1255
Training DefconModel on 63,480 samples (14 features, tuned selection)...
  Target mean: 5.8964
  MAE: 2.3565
Training BonusModel (Monte Carlo, 5000 sims)...
  Estimating baseline BPS from stats (no actual BPS data)
Training BaselineBPSModel on 45343 samples...
  Features used: 30
  Avg baseline BPS: 18.7
  MAE: 0.69
  Loaded FPL availability for 1612 players
Training CardsModel on 63,480 samples (19 features)...
  Yellow cards: 395/63,480 (0.6%)
  LogLoss: 0.0221
Training SavesModel on 4,397 GK samples (13 features, tuned selection)...
  Mean saves/90: 2.99
  MAE: 1.437

PREDICTING GW31 (2025/2026)
Found 520 active players with historical data
  Filtered out 77 unavailable players (injured

In [4]:
# =============================================================================
# STEP 3: View Top Players
# =============================================================================
# Top 20 by expected points with full prediction breakdown
display_cols = [
    'player_name', 'team', 'fpl_position', 'opponent', 'is_home',
    'pred_minutes', 'pred_exp_goals', 'pred_exp_assists', 
    'pred_cs_prob', 'pred_2plus_conceded', 'pred_4plus_conceded',
    'pred_defcon_prob', 'pred_exp_saves',
    'pred_yellow_prob', 'pred_red_prob',
    'pred_bonus', 'exp_total_pts'
]
available_cols = [c for c in display_cols if c in predictions.columns]
predictions.loc[predictions['fpl_position']!="GK"].nlargest(20, 'exp_total_pts')[available_cols].round(2)

,player_name,team,fpl_position,opponent,is_home,pred_minutes,pred_exp_goals,pred_exp_assists,pred_cs_prob,pred_2plus_conceded,pred_4plus_conceded,pred_defcon_prob,pred_exp_saves,pred_yellow_prob,pred_red_prob,pred_bonus,exp_total_pts
227,Cole Palmer,Chelsea,MID,Everton,0,83.070000,0.49,0.23,0.33,0.30,0.03,0.02,0.0,0.00,0.00,0.69,6.20
33,Mohamed Salah,Liverpool,MID,Brighton,0,88.019997,0.40,0.19,0.35,0.28,0.02,0.00,0.0,0.00,0.00,0.59,5.48
32,Raul Jiménez,Fulham,FWD,Burnley,1,82.110001,0.50,0.14,0.38,0.25,0.02,0.01,0.0,0.00,0.00,1.03,5.41
266,Lewis Hall,Newcastle United,DEF,Sunderland,1,85.750000,0.07,0.11,0.38,0.25,0.02,0.12,0.0,0.01,0.00,1.00,5.22
248,Malick Thiaw,Newcastle United,DEF,Sunderland,1,89.629997,0.16,0.06,0.38,0.25,0.02,0.31,0.0,0.03,0.00,0.25,5.22
145,Bruno Guimaraes,Newcastle United,MID,Sunderland,1,89.309998,0.23,0.28,0.38,0.25,0.02,0.23,0.0,0.00,0.00,0.40,5.20
290,Igor Thiago,Brentford,FWD,Leeds,0,89.230003,0.46,0.11,0.26,0.38,0.05,0.02,0.0,0.11,0.01,0.97,5.03
49,Harry Wilson,Fulham,MID,Burnley,1,85.860001,0.30,0.22,0.38,0.25,0.02,0.01,0.0,0.00,0.00,0.40,4.98
43,Bruno Fernandes,Manchester United,MID,Bournemouth,0,88.910004,0.29,0.20,0.23,0.43,0.06,0.09,0.0,0.00,0.00,0.49,4.92
170,Anthony Gordon,Newcastle United,MID,Sunderland,1,81.599998,0.31,0.19,0.38,0.25,0.02,0.00,0.0,0.00,0.00,0.36,4.90


In [9]:
# =============================================================================
# Interactive Distribution (D3.js) — opens in browser
# =============================================================================
from src.viz import generate_distribution_html, generate_mobile_html
import webbrowser, os

# Get eval metrics from training
viz_metrics = pipeline.get_viz_metrics()

# Desktop version (ridge plot)
html_path = generate_distribution_html(
    predictions,
    pipeline.last_simulations,
    output_path='distributions.html',
    top_n=100,
    gameweek=31,
    metrics=viz_metrics,
)

# Mobile version (card layout)
mobile_path = generate_mobile_html(
    predictions,
    pipeline.last_simulations,
    output_path='distributions_mobile.html',
    top_n=100,
    gameweek=31,
    metrics=viz_metrics,
)

# Save full run (predictions, simulations, params, metrics)
run_dir = pipeline.save_run(
    predictions,
    gameweek=31,
    season='2025/2026',
    description='baseline: full tune, 5k sims, 25/26 holdout inc-bonus MAE',
)

webbrowser.open('file://' + os.path.realpath(html_path))

Distribution visualization saved to: distributions.html
Mobile distribution visualization saved to: distributions_mobile.html

Run saved to: data\runs\gw31_20260319_100445
  predictions.csv: 351 players
  simulations: 5000 sims
  tuned_params.json: 6 models
  FPL MAE: ex-bonus=1.7236, inc-bonus=1.7794


True

In [7]:
# =============================================================================
# FPL Points Breakdown: Predicted vs Actual by Category
# Shows where the model over/under-predicts in FPL points terms
# =============================================================================
breakdown = pipeline.points_breakdown()
display(breakdown.style.format({
    'pred_value_avg': '{:.4f}',
    'pred_pts_avg': '{:.4f}',
    'actual_value_avg': '{:.4f}',
    'actual_pts_avg': '{:.4f}',
    'pts_diff': '{:+.4f}',
    'abs_pts_diff': '{:.4f}',
}, na_rep='—').set_caption('FPL Points Breakdown: Predicted vs Actual (test set averages per player-match)'))

c:\Users\dpfin\OneDrive\Desktop\projecting_fpl_v2\src\pipeline.py:2143: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  result = pd.concat([result, totals], ignore_index=True)


,category,pred_value_avg,pred_pts_avg,actual_value_avg,actual_pts_avg,pts_diff,abs_pts_diff
0,Appearance,67.6608,1.6795,66.0449,1.6928,-0.0133,0.0133
1,Goals,0.0946,0.4402,0.0861,0.3955,+0.0448,0.0448
2,Assists,0.0688,0.2063,0.0627,0.1880,+0.0184,0.0184
3,Clean Sheet,0.2096,0.3328,0.2271,0.4843,-0.1515,0.1515
4,Goals Conceded,1.3952,-0.1502,1.2393,-0.1346,-0.0156,0.0156
5,Saves,2.9379,0.0664,2.6667,0.0380,+0.0284,0.0284
6,Defcon,0.1663,0.1627,0.2291,0.2241,-0.0614,0.0614
7,Yellow Cards,0.0342,-0.0342,0.0422,-0.0422,+0.0080,0.0080
8,Red Cards,0.0027,-0.0080,0.0021,-0.0063,-0.0017,0.0017
9,Bonus,0.2205,0.2205,0.0000,0.0000,+0.2205,0.2205


In [8]:
importlib.reload(src.viz)

<module 'src.viz' from 'c:\\Users\\dpfin\\OneDrive\\Desktop\\projecting_fpl_v2\\src\\viz.py'>

In [1]:
# =============================================================================
# Defcon count distribution for a random 2025/26 gameweek
# =============================================================================
import matplotlib.pyplot as plt
import numpy as np

df = pipeline.df.copy()
season_df = df[df['season'] == '2025/2026']
available_gws = sorted(season_df['gameweek'].dropna().unique())
rng = np.random.default_rng()
chosen_gw = rng.choice(available_gws)

gw_df = season_df[season_df['gameweek'] == chosen_gw].copy()
gw_df = gw_df[gw_df['defcon'].notna() & (gw_df['minutes'] > 0)]

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)

groups = [
    ('DEF', gw_df[gw_df['is_def'] == 1]),
    ('MID', gw_df[gw_df['is_mid'] == 1]),
    ('All outfield (min > 0)', gw_df[gw_df['is_gk'] != 1]),
]

for ax, (label, sub) in zip(axes, groups):
    vals = sub['defcon'].values
    bins = np.arange(vals.min() - 0.5, vals.max() + 1.5, 1)
    ax.hist(vals, bins=bins, edgecolor='white', alpha=0.8, color='#58a6ff')
    ax.set_title(f'{label}  (n={len(vals)})', fontsize=13)
    ax.set_xlabel('Defcon count')
    ax.axvline(vals.mean(), color='#f85149', ls='--', lw=1.5, label=f'mean={vals.mean():.1f}')
    ax.legend(fontsize=10)

axes[0].set_ylabel('# Players')
fig.suptitle(f'Defcon counts — 2025/26 GW {int(chosen_gw)}', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

print(f"GW {int(chosen_gw)}: {len(gw_df)} player-matches with minutes > 0")
print(f"DEF defcon: mean={gw_df.loc[gw_df['is_def']==1,'defcon'].mean():.1f}, "
      f"median={gw_df.loc[gw_df['is_def']==1,'defcon'].median():.0f}, "
      f"max={gw_df.loc[gw_df['is_def']==1,'defcon'].max():.0f}")
print(f"MID defcon: mean={gw_df.loc[gw_df['is_mid']==1,'defcon'].mean():.1f}, "
      f"median={gw_df.loc[gw_df['is_mid']==1,'defcon'].median():.0f}, "
      f"max={gw_df.loc[gw_df['is_mid']==1,'defcon'].max():.0f}")

NameError: name 'pipeline' is not defined

In [ ]:
# =============================================================================
# Goals & Assists count distributions for a random 2025/26 gameweek
# + overdispersion check (variance / mean) across ALL 2025/26 data
# =============================================================================
import matplotlib.pyplot as plt
import numpy as np

df = pipeline.df.copy()
season_df = df[(df['season'] == '2025/2026') & (df['minutes'] > 0)]
available_gws = sorted(season_df['gameweek'].dropna().unique())
rng = np.random.default_rng()
chosen_gw = rng.choice(available_gws)
gw_df = season_df[season_df['gameweek'] == chosen_gw]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# --- Row 1: Goals ---
for ax, (label, sub) in zip(axes[0], [
    ('DEF', gw_df[gw_df['is_def'] == 1]),
    ('MID', gw_df[gw_df['is_mid'] == 1]),
    ('FWD', gw_df[gw_df['is_fwd'] == 1]),
]):
    vals = sub['goals'].fillna(0).values
    bins = np.arange(-0.5, max(vals.max(), 3) + 1.5, 1)
    ax.hist(vals, bins=bins, edgecolor='white', alpha=0.8, color='#3fb950')
    ax.set_title(f'Goals — {label}  (n={len(vals)})', fontsize=12)
    ax.set_xlabel('Goal count')
    ax.axvline(vals.mean(), color='#f85149', ls='--', lw=1.5, label=f'mean={vals.mean():.2f}')
    ax.legend(fontsize=10)

axes[0][0].set_ylabel('# Players')

# --- Row 2: Assists ---
for ax, (label, sub) in zip(axes[1], [
    ('DEF', gw_df[gw_df['is_def'] == 1]),
    ('MID', gw_df[gw_df['is_mid'] == 1]),
    ('FWD', gw_df[gw_df['is_fwd'] == 1]),
]):
    vals = sub['assists'].fillna(0).values
    bins = np.arange(-0.5, max(vals.max(), 3) + 1.5, 1)
    ax.hist(vals, bins=bins, edgecolor='white', alpha=0.8, color='#d29922')
    ax.set_title(f'Assists — {label}  (n={len(vals)})', fontsize=12)
    ax.set_xlabel('Assist count')
    ax.axvline(vals.mean(), color='#f85149', ls='--', lw=1.5, label=f'mean={vals.mean():.2f}')
    ax.legend(fontsize=10)

axes[1][0].set_ylabel('# Players')

fig.suptitle(f'Goals & Assists counts — 2025/26 GW {int(chosen_gw)}', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

# --- Overdispersion check across ALL 2025/26 data ---
print("=" * 65)
print("OVERDISPERSION CHECK — 2025/26, all GWs, minutes > 0")
print("=" * 65)
print(f"{'Stat':<12} {'Position':<10} {'Mean':>8} {'Variance':>10} {'Var/Mean':>10} {'Overdispersed?'}")
print("-" * 65)

outfield = season_df[season_df['is_gk'] != 1]
for stat in ['goals', 'assists', 'defcon']:
    for pos_label, mask in [('DEF', outfield['is_def']==1),
                             ('MID', outfield['is_mid']==1),
                             ('FWD', outfield['is_fwd']==1),
                             ('All OF', outfield.index == outfield.index)]:
        vals = outfield.loc[mask, stat].dropna().values
        m, v = vals.mean(), vals.var()
        ratio = v / m if m > 0 else float('nan')
        flag = "YES" if ratio > 1.5 else ("marginal" if ratio > 1.1 else "no")
        print(f"{stat:<12} {pos_label:<10} {m:>8.3f} {v:>10.3f} {ratio:>10.2f}   {flag}")
    print()